# Teknik Validasi Silang dan Evaluasi Model

Catatan ini menguraikan metodologi tingkat lanjut untuk mengevaluasi keandalan dan kemampuan generalisasi model *machine learning*. Pemisahan data latih dan data uji (*train-test split*) konvensional sering kali tidak cukup karena sangat bergantung pada bagaimana data tersebut dipecah secara acak.

Untuk mengatasi variansi dan bias dari pemisahan data tunggal, kita menggunakan teknik **Validasi Silang (*Cross-Validation*)**. Modul ini berfokus pada berbagai strategi validasi silang yang disediakan oleh `scikit-learn` dan kapan waktu yang tepat untuk menggunakannya.

#### **Tujuan Pembelajaran**
* Memahami kelemahan dari metode *Train-Test Split* standar.
* Mengimplementasikan **K-Fold Cross-Validation** untuk evaluasi model yang komprehensif.
* Menggunakan **Stratified K-Fold** untuk menjaga proporsi kelas pada masalah klasifikasi.
* Mengaplikasikan strategi komputasi intensif seperti **Leave-One-Out Cross-Validation (LOOCV)** pada dataset berukuran kecil.
* Memahami dan menggunakan **ShuffleSplit** untuk kontrol proporsi partisi yang lebih fleksibel.
* Menerapkan **GroupKFold** untuk mencegah kebocoran informasi (*data leakage*) pada himpunan data yang memiliki dependensi grup (seperti data rekam medis pasien).

In [53]:
# Memuat pustaka komputasi numerik dan visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Memuat dataset
from sklearn.datasets import load_iris, make_blobs

# Memuat modul strategi Validasi Silang (Cross-Validation)
from sklearn.model_selection import (
    cross_val_score,
    KFold,
    StratifiedKFold,
    LeaveOneOut,
    ShuffleSplit,
    GroupKFold
)

# Memuat algoritma klasifikasi sebagai model pengujian
from sklearn.linear_model import LogisticRegression

# Konfigurasi visualisasi
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print("Pustaka untuk Teknik Validasi Silang berhasil dimuat.")

Pustaka untuk Teknik Validasi Silang berhasil dimuat.


## 1. K-Fold Cross-Validation

*K-Fold Cross-Validation* adalah pendekatan fundamental di mana himpunan data dibagi menjadi $K$ bagian (lipatan) yang berukuran sama. Model kemudian dilatih sebanyak $K$ kali. Pada setiap iterasinya, satu lipatan dialokasikan sebagai himpunan uji (*test set*), dan sisa $K-1$ lipatan digabungkan sebagai himpunan latih (*train set*).

**Keunggulan:**
Setiap sampel data akan tepat satu kali menjadi bagian dari himpunan uji, sehingga estimasi akurasi model menjadi jauh lebih stabil dan objektif dibandingkan pemisahan tunggal.

In [54]:
# Memuat dataset Iris
iris = load_iris()
X, y = iris.data, iris.target

# Inisialisasi model baseline
logreg = LogisticRegression(max_iter=1000)

# 1. K-Fold Bawaan (Tanpa Pengacakan)
# Jika K=5, data dipotong berurutan 20% awal, 20% kedua, dst.
kfold_standar = KFold(n_splits=5)
skor_standar = cross_val_score(logreg, X, y, cv=kfold_standar)

# 2. K-Fold dengan Pengacakan (Shuffle)
# Mencegah bias jika data asal sudah terurut berdasarkan label
kfold_acak = KFold(n_splits=5, shuffle=True, random_state=42)
skor_acak = cross_val_score(logreg, X, y, cv=kfold_acak)

print("=== Komparasi K-Fold Cross-Validation ===")
print("Skor K-Fold (Tanpa Pengacakan) :", skor_standar)
print("Skor K-Fold (Dengan Pengacakan):", skor_acak)
print(f"Rata-rata Akurasi (Pengacakan) : {skor_acak.mean():.4f} (Standar Deviasi: {skor_acak.std():.4f})")

=== Komparasi K-Fold Cross-Validation ===
Skor K-Fold (Tanpa Pengacakan) : [1.         1.         0.86666667 0.93333333 0.83333333]
Skor K-Fold (Dengan Pengacakan): [1.         1.         0.93333333 0.96666667 0.96666667]
Rata-rata Akurasi (Pengacakan) : 0.9733 (Standar Deviasi: 0.0249)


## 2. Stratified K-Fold Cross-Validation

Pada kasus klasifikasi, menggunakan *K-Fold* standar bisa menjadi bencana jika urutan data tersusun berdasarkan kelas (misalnya baris 1-50 adalah Kelas A, baris 51-100 adalah Kelas B). Lipatan pertama mungkin hanya berisi Kelas A, sehingga model gagal belajar mendeteksi kelas lain.

*Stratified K-Fold* menyelesaikan masalah ini dengan memastikan bahwa proporsi kelas pada setiap lipatan secara matematis identik dengan proporsi kelas pada keseluruhan dataset. `scikit-learn` secara otomatis menggunakan *Stratified K-Fold* jika fungsi `cross_val_score` mendeteksi estimator klasifikasi.

In [55]:
# Inisialisasi Stratified K-Fold secara eksplisit
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
skor_skf = cross_val_score(logreg, X, y, cv=skf)

print("=== Stratified K-Fold ===")
print("Distribusi Kelas Aktual:", np.bincount(y))
print("Skor per Lipatan       :", skor_skf)
print(f"Rata-rata Akurasi      : {skor_skf.mean():.4f}")

=== Stratified K-Fold ===
Distribusi Kelas Aktual: [50 50 50]
Skor per Lipatan       : [1.   0.94 0.96]
Rata-rata Akurasi      : 0.9667


## 3. Leave-One-Out (LOOCV) dan ShuffleSplit

**Leave-One-Out Cross-Validation (LOOCV)**
Pendekatan ekstrem dari K-Fold di mana nilai $K$ didefinisikan sama persis dengan jumlah total observasi ($N$). Model dilatih sebanyak $N$ kali, dan setiap kali iterasi, tepat *satu* sampel tunggal diisolasi sebagai data uji. Sangat direkomendasikan untuk dataset berskala sangat kecil (misal: kurang dari 100 baris), namun sangat membebani komputasi untuk dataset besar.

**ShuffleSplit**
Strategi ini mengambil sampel himpunan latih dan uji secara acak dan independen pada setiap iterasi. Ukuran himpunan latih dan uji tidak harus terikat pada fraksi lipatan $1/K$, memberikan fleksibilitas penuh untuk menentukan jumlah iterasi dan ukuran partisi.

In [56]:
# 1. Leave-One-Out Cross Validation (LOOCV)
loocv = LeaveOneOut()
skor_loocv = cross_val_score(logreg, X, y, cv=loocv)

print("=== Leave-One-Out CV (LOOCV) ===")
print(f"Total Iterasi Pelatihan : {len(skor_loocv)} kali")
print(f"Estimasi Akurasi Rata-rata : {skor_loocv.mean():.4f}\n")


# 2. ShuffleSplit
# Mensimulasikan 10 iterasi dengan rasio latih 70% dan uji 30% independen
shuffle_split = ShuffleSplit(n_splits=10, train_size=0.7, test_size=0.3, random_state=42)
skor_ss = cross_val_score(logreg, X, y, cv=shuffle_split)

print("=== ShuffleSplit CV ===")
print("Skor 10 Iterasi Acak :", np.round(skor_ss, 3))
print(f"Rata-rata Akurasi    : {skor_ss.mean():.4f}")

=== Leave-One-Out CV (LOOCV) ===
Total Iterasi Pelatihan : 150 kali
Estimasi Akurasi Rata-rata : 0.9667

=== ShuffleSplit CV ===
Skor 10 Iterasi Acak : [1.    1.    0.911 0.956 0.933 0.978 0.956 0.956 1.    0.911]
Rata-rata Akurasi    : 0.9600


## 4. GroupKFold (Mencegah Kebocoran Informasi Grup)

Dalam beberapa eksperimen analitik, sampel data tidak terdistribusi secara independen (IID). Sebagai contoh, Anda memprediksi ekspresi wajah berdasarkan sampel gambar. Jika satu subjek (orang yang sama) memiliki 10 foto berbeda, dan foto-foto tersebut tersebar ke himpunan latih dan himpunan uji, model akan mengenali "wajah" subjek tersebut, bukan mengekstraksi fitur ekspresinya. Ini adalah bentuk *data leakage* laten.

*GroupKFold* dirancang sedemikian rupa agar seluruh sampel yang berasal dari grup (atau subjek) yang sama akan selalu dialokasikan secara utuh ke dalam himpunan latih saja, atau himpunan uji saja, tanpa terpotong antar-lipatan.

In [57]:
# Membuat dataset sintetis dengan indikator Grup (Dependensi antar baris)
X_grup, y_grup = make_blobs(n_samples=12, random_state=0)

# Simulasi: 12 baris data berasal dari 4 subjek (Grup 0, 1, 2, 3)
indikator_grup = np.array([0, 0, 0, 1, 1, 1, 1, 2, 2, 3, 3, 3])

# Inisialisasi GroupKFold dengan 3 Lipatan
group_kfold = GroupKFold(n_splits=3)

# Memasukkan array indikator grup ke dalam parameter 'groups'
skor_grup = cross_val_score(logreg, X_grup, y_grup, groups=indikator_grup, cv=group_kfold)

print("=== GroupKFold CV ===")
print("Vektor Indikator Grup :", indikator_grup)
print("Skor Evaluasi per Lipatan:", skor_grup)
print(f"Rata-rata Akurasi Obyektif : {skor_grup.mean():.4f}")

=== GroupKFold CV ===
Vektor Indikator Grup : [0 0 0 1 1 1 1 2 2 3 3 3]
Skor Evaluasi per Lipatan: [0.75       0.6        0.66666667]
Rata-rata Akurasi Obyektif : 0.6722


## Kesimpulan

Evaluasi menggunakan pemisahan tunggal (1 kali *train-test split*) sangat dilarang dalam pengujian model saintifik karena margin kebetulannya sangat tinggi. Gunakan metode yang sesuai dengan struktur topologi data Anda:

| Metode Evaluasi | Indikasi Penggunaan Utama |
| :--- | :--- |
| **K-Fold / K-Fold (Shuffle)** | Algoritma *Regression* atau dataset numerik kontinu skala menengah hingga besar. |
| **Stratified K-Fold** | Algoritma *Classification*. Mutlak diperlukan jika distribusi antar-kelas tidak seimbang (*imbalanced*). |
| **Leave-One-Out (LOOCV)** | Dataset klinis, riset, atau operasional berskala sangat kecil ($N < 100$). |
| **ShuffleSplit** | Eksperimentasi fleksibel di mana durasi komputasi K-Fold tidak memungkinkan untuk himpunan data raksasa. |
| **GroupKFold** | Rekam medis multi-kunjungan, pengukuran rentet temporal, atau gambar multi-pose dari identitas fisik yang sama. |